### Model Training: Customer Churn Prediction

In [1]:
import pandas as pd
import datetime
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard, ReduceLROnPlateau
from tensorflow.keras.metrics import Precision, Recall, AUC

#### Load the Processed Data

In [2]:
# Load Processed Data
X_train = pd.read_csv('../data/X_train_processed.csv')
X_val = pd.read_csv('../data/X_val_processed.csv')

# Flatten y dataframes to Series
y_train = pd.read_csv('../data/y_train.csv')['Exited']
y_val = pd.read_csv('../data/y_val.csv')['Exited']

#### Build and Compile the ANN Architecture

In [3]:
# Build ANN Model
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,521 (13.75 KB)

 Trainable params: 3,521 (13.75 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Compile Model
model.compile(
    loss='binary_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall'), AUC(name='roc_auc')]
)

#### Training the Model

In [5]:
# Callbacks
log_dir = '../logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
callbacks = [
    TensorBoard(log_dir=log_dir, histogram_freq=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

In [6]:
# Applying class weights to penalize the model for missing the minority class (actual churners).
class_weights = {0: 1., 1: 2.5}

print("Training model...")
history = model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val), 
    epochs=100, 
    batch_size=32,
    callbacks=callbacks,
    class_weight=class_weights
)

Training model...
Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7559 - loss: 0.7436 - precision: 0.4214 - recall: 0.5307 - roc_auc: 0.7521 - val_accuracy: 0.8000 - val_loss: 0.4568 - val_precision: 0.5087 - val_recall: 0.5368 - val_roc_auc: 0.7856 - learning_rate: 0.0010
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8045 - loss: 0.6629 - precision: 0.5173 - recall: 0.6089 - roc_auc: 0.8142 - val_accuracy: 0.8238 - val_loss: 0.4335 - val_precision: 0.5675 - val_recall: 0.5675 - val_roc_auc: 0.8082 - learning_rate: 0.0010
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8258 - loss: 0.6200 - precision: 0.5634 - recall: 0.6442 - roc_auc: 0.8403 - val_accuracy: 0.8163 - val_loss: 0.4315 - val_precision: 0.5430 - val_recall: 0.6196 - val_roc_auc: 0.8301 - learning_rate: 0.0010
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8330 - loss: 0.5879 - precision: 0.5787 - recall: 0.6626 - roc_auc: 0.8583 - val_accuracy:

In [7]:
# Save Final Model
model.save('../models/ann_model.keras')
print("Model saved to ../models/ann_model.keras")

Model saved to ../models/ann_model.keras


In [ ]:
# Load TensorBoard Extension
%load_ext tensorboard

In [ ]:
# Visualize the Logs
%tensorboard --logdir $log_dir